In [1]:
import sys
from pathlib import Path

ROOT = Path("./arcface_torch")
sys.path.insert(0, str(ROOT))

### Set Configs

In [2]:
%%bash
cat > ./arcface_torch/configs/faces_webface_vit_s.py <<'EOF'
from easydict import EasyDict as edict

config = edict()
config.margin_list = (1.0, 0.0, 0.4)
config.network = "vit_s"
config.resume = False
config.output = None
config.embedding_size = 512
config.sample_rate = 1.0
config.fp16 = True
config.momentum = 0.9
config.weight_decay = 1e-4
config.batch_size = 128
config.lr = 0.1
config.verbose = 2000
config.dali = False

config.rec = "./data/faces_webface_112x112"
config.num_classes = 10572
config.num_image = 490623
config.num_epoch = 20
config.warmup_epoch = 0
config.val_targets = ['lfw']
EOF

### Train

In [1]:
# import numpy as np
# np.long = np.int_
# np.ulong = np.uint

# if not hasattr(np, "bool"):
#     np.bool = bool
    
# import mxnet as mx  

# import argparse
# import logging
# import os
# from datetime import datetime

# import numpy as np
# import torch
# from backbones import get_model
# from dataset import get_dataloader
# from losses import CombinedMarginLoss
# from lr_scheduler import PolynomialLRWarmup
# from partial_fc_v2 import PartialFC_V2
# from torch import distributed
# from torch.utils.data import DataLoader
# from torch.utils.tensorboard import SummaryWriter
# from utils.utils_callbacks import CallBackLogging, CallBackVerification
# from utils.utils_config import get_config
# from utils.utils_distributed_sampler import setup_seed
# from utils.utils_logging import AverageMeter, init_logging
# from torch.distributed.algorithms.ddp_comm_hooks.default_hooks import fp16_compress_hook

In [4]:
rank = 0
local_rank = 0
world_size = 1

In [5]:
distributed.init_process_group(
    backend="nccl",
    init_method="tcp://127.0.0.1:12584",
    rank=rank,
    world_size=world_size,
)

In [6]:
torch.backends.cudnn.benchmark = True

In [7]:
parser = argparse.ArgumentParser(
    description="Distributed Arcface Training in Pytorch")
parser.add_argument("config", type=str, help="py config file")

_StoreAction(option_strings=[], dest='config', nargs=None, const=None, default=None, type=<class 'str'>, choices=None, required=True, help='py config file', metavar=None)

In [8]:
args = parser.parse_args(["configs/faces_webface_vit_s.py"])

In [2]:
# cfg = get_config(args.config)
# setup_seed(seed=cfg.seed, cuda_deterministic=False)
# torch.cuda.set_device(local_rank)
# os.makedirs(cfg.output, exist_ok=True)
# init_logging(rank, cfg.output)

# summary_writer = (
#     SummaryWriter(log_dir=os.path.join(cfg.output, "tensorboard"))
#     if rank == 0
#     else None
# )

# wandb_logger = None
# train_loader = get_dataloader(
#     cfg.rec,
#     local_rank,
#     cfg.batch_size,
#     cfg.dali,
#     cfg.dali_aug,
#     cfg.seed,
#     cfg.num_workers
# )

# backbone = get_model(
#     cfg.network, dropout=0.0, fp16=cfg.fp16, num_features=cfg.embedding_size).cuda()

# backbone = torch.nn.parallel.DistributedDataParallel(
#     module=backbone, broadcast_buffers=False, device_ids=[local_rank], bucket_cap_mb=16,
#     find_unused_parameters=True)
# backbone.register_comm_hook(None, fp16_compress_hook)
# backbone.train()

# backbone._set_static_graph()
# margin_loss = CombinedMarginLoss(
#     64,
#     cfg.margin_list[0],
#     cfg.margin_list[1],
#     cfg.margin_list[2],
#     cfg.interclass_filtering_threshold
# )
# module_partial_fc = PartialFC_V2(
#     margin_loss, cfg.embedding_size, cfg.num_classes,
#     cfg.sample_rate, False)
# module_partial_fc.train().cuda()

# opt = torch.optim.SGD(
#     params=[{"params": backbone.parameters()}, {"params": module_partial_fc.parameters()}],
#     lr=cfg.lr, momentum=0.9, weight_decay=cfg.weight_decay)

# cfg.total_batch_size = cfg.batch_size * world_size
# cfg.warmup_step = cfg.num_image // cfg.total_batch_size * cfg.warmup_epoch
# cfg.total_step = cfg.num_image // cfg.total_batch_size * cfg.num_epoch

# lr_scheduler = PolynomialLRWarmup(
#     optimizer=opt,
#     warmup_iters=cfg.warmup_step,
#     total_iters=cfg.total_step)

# start_epoch = 0
# global_step = 0

# for key, value in cfg.items():
#     num_space = 25 - len(key)
#     logging.info(": " + key + " " * num_space + str(value))

# callback_verification = CallBackVerification(
#     val_targets=cfg.val_targets, rec_prefix=cfg.rec, 
#     summary_writer=summary_writer, wandb_logger = wandb_logger
# )
# callback_logging = CallBackLogging(
#     frequent=cfg.frequent,
#     total_step=cfg.total_step,
#     batch_size=cfg.batch_size,
#     start_step = global_step,
#     writer=summary_writer
# )

# loss_am = AverageMeter()
# amp = torch.cuda.amp.grad_scaler.GradScaler(growth_interval=100)

In [3]:
# for epoch in range(start_epoch, cfg.num_epoch):

#     if isinstance(train_loader, DataLoader):
#         train_loader.sampler.set_epoch(epoch)
#     for _, (img, local_labels) in enumerate(train_loader):
#         global_step += 1
#         local_embeddings = backbone(img)
#         loss: torch.Tensor = module_partial_fc(local_embeddings, local_labels)

#         if cfg.fp16:
#             amp.scale(loss).backward()
#             if global_step % cfg.gradient_acc == 0:
#                 amp.unscale_(opt)
#                 torch.nn.utils.clip_grad_norm_(backbone.parameters(), 5)
#                 amp.step(opt)
#                 amp.update()
#                 opt.zero_grad()
#         else:
#             loss.backward()
#             if global_step % cfg.gradient_acc == 0:
#                 torch.nn.utils.clip_grad_norm_(backbone.parameters(), 5)
#                 opt.step()
#                 opt.zero_grad()
#         lr_scheduler.step()

#         with torch.no_grad():
#             if wandb_logger:
#                 wandb_logger.log({
#                     'Loss/Step Loss': loss.item(),
#                     'Loss/Train Loss': loss_am.avg,
#                     'Process/Step': global_step,
#                     'Process/Epoch': epoch
#                 })
                
#             loss_am.update(loss.item(), 1)
#             callback_logging(global_step, loss_am, epoch, cfg.fp16, lr_scheduler.get_last_lr()[0], amp)

#             if global_step % cfg.verbose == 0 and global_step > 0:
#                 callback_verification(global_step, backbone)

#     if cfg.save_all_states:
#         checkpoint = {
#             "epoch": epoch + 1,
#             "global_step": global_step,
#             "state_dict_backbone": backbone.module.state_dict(),
#             "state_dict_softmax_fc": module_partial_fc.state_dict(),
#             "state_optimizer": opt.state_dict(),
#             "state_lr_scheduler": lr_scheduler.state_dict()
#         }
#         torch.save(checkpoint, os.path.join(cfg.output, f"checkpoint_gpu_{rank}.pt"))

#     if rank == 0:
#         path_module = os.path.join(cfg.output, "model.pt")
#         torch.save(backbone.module.state_dict(), path_module)

#         if wandb_logger and cfg.save_artifacts:
#             artifact_name = f"{run_name}_E{epoch}"
#             model = wandb.Artifact(artifact_name, type='model')
#             model.add_file(path_module)
#             wandb_logger.log_artifact(model)
            
#     if cfg.dali:
#         train_loader.reset()